In [ ]:
!pip install "psycopg[binary]==3.3.6" "python-dotenv==1.2.4"

This installs the two Python packages the lesson uses, at the exact versions listed in Section 6: `psycopg` (the PostgreSQL driver that sends our commands in Structured Query Language (SQL) and returns rows) and `python-dotenv` (which reads our connection settings from a `.env` file).

In [ ]:
# Bring in the tools we need: standard-library helpers, the PostgreSQL driver
# (psycopg), its safe SQL builder (sql), and the .env loader.
import os
from decimal import Decimal

import psycopg
from psycopg import sql
from dotenv import load_dotenv

We import the pieces we use throughout: `os` and `Decimal` from Python's standard library, the `psycopg` driver that talks to PostgreSQL, the `sql` helper that safely quotes names, and `load_dotenv` to read our settings file.

In [ ]:
# Read the connection settings from the shared .env file two folders up
# (the section folder, then the course root).
load_dotenv("../../.env")

PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD") or None  # an empty value means "no password"
PGDATABASE = os.getenv("PGDATABASE", "university")
PGSSLMODE = os.getenv("PGSSLMODE", "prefer")

print(f"Connecting to {PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE} (sslmode={PGSSLMODE})")

We load the connection details from one shared `.env` file so no password is ever written into the notebook. Each setting falls back to a sensible default (`localhost`, port `5432`, user `postgres`, database `university`), so the lesson still runs if a value is missing. The path `../../.env` goes up from this lesson folder to the section folder and then to the course root — that is why we start Jupyter from inside this lesson folder.

In [ ]:
# Connect to the built-in `postgres` database (a server hosts many databases)
# and create our `university` database only if it doesn't exist yet.
conn_args = dict(host=PGHOST, port=PGPORT, user=PGUSER, password=PGPASSWORD, sslmode=PGSSLMODE, autocommit=True)

with psycopg.connect(dbname="postgres", **conn_args) as admin:
    exists = admin.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PGDATABASE,)).fetchone()
    if exists:
        print(f"Database '{PGDATABASE}' already exists.")
    else:
        # Names can't be passed as %s values, so we quote the identifier safely.
        admin.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(PGDATABASE)))
        print(f"Database '{PGDATABASE}' created.")

This step is a small branch, so it is worth showing its shape:

```mermaid
graph TD
    S["Connect to the<br/>postgres database"] --> Q{"Does 'university'<br/>already exist?"}
    Q -->|yes| E["Print: already exists"]
    Q -->|no| C["Run CREATE DATABASE<br/>and print: created"]
    style S fill:#e1f5ff,stroke:#0288d1,color:#01579b
    style Q fill:#fff9c4,stroke:#f9a825,color:#7f6000
    style E fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
    style C fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
```

- **Input:** the connection settings we just loaded, and the database name `university`.
- **What happens:** we open a connection to the `postgres` maintenance database, ask whether `university` already exists, and create it only if it does not.
- **Output:** the `university` database exists on the server, and you see a short confirmation line.
- **Why:** the notebook stays safe to re-run, and every step after this connects to `university` instead of the maintenance database.

In [ ]:
# Open the real lesson connection to the `university` database, with
# autocommit on so each statement takes effect immediately.
conn = psycopg.connect(dbname=PGDATABASE, **conn_args)


# A small helper that runs a query and prints the rows as an aligned table,
# so our output looks like a real psql session. We use it in almost every step.
def show(query, params=None):
    """Run a query and print the result as a psql-style table."""
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description is None:              # CREATE/INSERT return no rows
            print(cur.statusmessage)
            return
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    fmt = lambda v: "NULL" if v is None else ("t" if v is True else "f" if v is False else str(v))
    cells = [[fmt(v) for v in r] for r in rows]
    numeric = [bool(rows) and all(v is None or (isinstance(v, (int, float, Decimal)) and not isinstance(v, bool)) for v in col)
               for col in zip(*rows)] or [False] * len(cols)
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    print(" | ".join(c.center(w) for c, w in zip(cols, widths)).rstrip())
    print("-+-".join("-" * w for w in widths))
    for r in cells:
        print(" | ".join(v.rjust(w) if numeric[i] else v.ljust(w) for i, (v, w) in enumerate(zip(r, widths))).rstrip())
    print(f"({len(rows)} row{'' if len(rows) == 1 else 's'})")


# Quick check that we landed in the right database.
show("SELECT current_database() AS connected_to")

- **Input:** a SQL query (a string) and, optionally, the values that fill its `%s` placeholders.
- **What happens:** we run the query on our connection and look at the result. If the statement returned no rows (such as `CREATE` or `INSERT`), we print the server's status message instead. Otherwise we read the column names and every row, then line them up into a fixed-width table, right-aligning numbers the way `psql` does.
- **Output:** the query result printed as a readable table, with a count of the rows.
- **Why:** it lets us spend our attention on SQL, not on formatting. Writing this once means every step below is one short `show(...)` call.

This is the one function we keep, because almost every step calls it (Section 6 lists the packages it relies on). You can run any of these queries directly in `psql` or pgAdmin and get the same rows.

In [ ]:
# Step 1: look around the server before we create anything. Confirm which
# database, role and schema we're in, and what the search path is.
show("""
    SELECT current_database() AS database,
           current_user       AS role,
           current_schema()   AS schema
""")

show("SHOW search_path")

# List the schemas in this database, ignoring PostgreSQL's internal ones.
show("""
    SELECT nspname AS schema
    FROM pg_namespace
    WHERE nspname !~ '^pg_' AND nspname <> 'information_schema'
    ORDER BY nspname
""")

Before creating anything, we look around. PostgreSQL organizes objects as **cluster > database > schema > table**: your connection is attached to exactly one database, and tables live in a *schema* inside it — by default the schema named `public`. The `search_path` setting tells PostgreSQL which schemas to look in when you write a bare table name such as `students`.

`current_schema()` returns `public` because our `search_path` is `"$user", public` and there is no schema named after our role. Every table this lesson creates goes into `public`.

In [ ]:
# Step 2: create the `students` table. DROP first so re-running the notebook
# starts clean, then CREATE the table with one typed column per fact.
conn.execute("DROP TABLE IF EXISTS students")

conn.execute("""
    CREATE TABLE students (
        student_id      TEXT,      -- business key such as STU001
        name            TEXT,
        email           TEXT,      -- some students have no email on file (NULL)
        course          TEXT,
        grade           INTEGER,   -- whole number 0-100
        enrollment_date DATE,      -- a calendar date, with no time part
        status          TEXT,      -- active / inactive / graduated
        scholarship     BOOLEAN,   -- true or false
        graduation_date DATE       -- NULL until the student graduates
    )
""")
print("students table created")

A table is a named set of rows that all share the same columns. `CREATE TABLE` lists each column with a **data type**, and PostgreSQL enforces it: you cannot store the word `abc` in an `INTEGER` column. We run `DROP TABLE IF EXISTS` first so the notebook is safe to run again and again. We deliberately add no constraints yet, to keep this first lesson focused on creating and reading data.

In [ ]:
# Ask PostgreSQL to describe the table it just built.
show("""
    SELECT ordinal_position AS pos, column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = 'students'
    ORDER BY ordinal_position
""")

`information_schema.columns` is a standard catalog view that lists every column of every table, so it is a reliable way to confirm what we built. In `psql` the same check is the one-letter command `\d students`; in pgAdmin, expand *Databases > university > Schemas > public > Tables* and right-click the table.

In [ ]:
# Step 3: our 25 sample students. Each tuple holds the nine column values in
# order; a Python None becomes SQL NULL.
students = [
    ("STU001", "Alice Johnson", "alice.johnson@university.example", "Computer Science", 95, "2024-09-01", "active", True, None),
    ("STU002", "Bob Smith", "bob.smith@university.example", "Mathematics", 78, "2024-09-01", "active", False, None),
    ("STU003", "Charlie Brown", "charlie.brown@university.example", "Physics", 55, "2024-09-02", "active", False, None),
    ("STU004", "Diana Prince", "diana.prince@university.example", "English", 48, "2024-09-01", "active", False, None),
    ("STU005", "Eve Torres", "eve.torres@university.example", "Biology", 88, "2024-09-03", "active", False, None),
    ("STU006", "Frank Castle", None, "Mathematics", 52, "2024-09-01", "inactive", False, None),
    ("STU007", "Grace Hopper", "grace.hopper@university.example", "Computer Science", 91, "2024-09-02", "active", True, None),
    ("STU008", "Hank Pym", "hank.pym@university.example", "Physics", 73, "2024-09-01", "active", False, None),
    ("STU009", "Ivan Petrov", "ivan.petrov@university.example", "Computer Science", 84, "2024-09-03", "active", False, None),
    ("STU010", "Julia Child", "julia.child@university.example", "English", 90, "2024-09-02", "graduated", True, "2025-06-30"),
    ("STU011", "Karl Marx", "karl.marx@university.example", "Mathematics", 67, "2024-09-01", "active", False, None),
    ("STU012", "Laura Palmer", None, "Biology", 45, "2024-09-02", "active", False, None),
    ("STU013", "Marco Polo", "marco.polo@university.example", "Physics", 82, "2024-09-03", "active", False, None),
    ("STU014", "Nina Simone", "nina.simone@university.example", "English", 76, "2024-09-01", "active", False, None),
    ("STU015", "Oscar Wilde", "oscar.wilde@university.example", "Mathematics", 93, "2024-09-02", "active", True, None),
    ("STU016", "Pia Zadora", "pia.zadora@university.example", "Biology", 71, "2024-09-01", "active", False, None),
    ("STU017", "Quincy Adams", "quincy.adams@university.example", "Computer Science", 87, "2024-09-03", "active", False, None),
    ("STU018", "Rosa Parks", None, "English", 58, "2024-09-02", "inactive", False, None),
    ("STU019", "Sam Wilson", "sam.wilson@university.example", "Physics", 98, "2024-09-01", "active", True, None),
    ("STU020", "Tina Turner", "tina.turner@university.example", "Mathematics", 79, "2024-09-03", "active", False, None),
    ("STU021", "Uma Thurman", "uma.thurman@university.example", "Computer Science", 62, "2024-09-02", "active", False, None),
    ("STU022", "Vera Wang", "vera.wang@university.example", "Biology", 85, "2024-09-01", "graduated", False, "2025-06-30"),
    ("STU023", "Walt Disney", "walt.disney@university.example", "Physics", 56, "2024-09-02", "active", False, None),
    ("STU024", "Xena Warrior", "xena.warrior@university.example", "English", 94, "2024-09-03", "active", True, None),
    ("STU025", "Yusuf Islam", "yusuf.islam@university.example", "Computer Science", 70, "2024-09-01", "active", False, None),
]

Here is the input for the whole lesson: 25 students, one tuple per row, with the values in the same order as the table's columns. Writing `None` where a value is missing is how we say "unknown" — it becomes SQL `NULL`, which we handle properly in Step 7.

In [ ]:
# Insert all 25 rows in a single executemany call. The %s placeholders carry
# the values separately from the SQL text, so they can never be treated as code.
with conn.cursor() as cur:
    cur.executemany(
        """INSERT INTO students
           (student_id, name, email, course, grade, enrollment_date, status, scholarship, graduation_date)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)""",
        students,
    )

# Confirm the rows landed.
show("SELECT count(*) AS students_loaded FROM students")

- **Input:** the 25 tuples we just defined.
- **What happens:** `executemany` runs the same `INSERT` once per tuple, sending the values as parameters; PostgreSQL checks each value against its column type and stores the row.
- **Output:** 25 rows now live in the `students` table.
- **Why:** loading all rows in one call is both faster and safer than building a giant SQL string, and it keeps the data out of the SQL text.

The values never touch the SQL string — they travel alongside it:

```mermaid
graph LR
    P["Python list<br/>(25 tuples)"] --> X["executemany<br/>with %s placeholders"]
    X --> G["PostgreSQL<br/>checks each type"]
    G --> T["students table<br/>(25 rows)"]
    style P fill:#e1f5ff,stroke:#0288d1,color:#01579b
    style X fill:#ffe0b2,stroke:#ef6c00,color:#7f3f00
    style G fill:#ffe0b2,stroke:#ef6c00,color:#7f3f00
    style T fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
```

In [ ]:
# Step 4: read data with SELECT. Start with every column, first five rows.
show("SELECT * FROM students ORDER BY student_id LIMIT 5")

`SELECT` reads data without changing it. `*` means "all columns". We add `ORDER BY student_id` because a table has **no guaranteed row order** — without it, the same query could return rows in any order, so we always sort to make the output repeatable. `LIMIT 5` caps the result to five rows.

In [ ]:
# Choose just the columns we care about, and rename them for the reader.
show("""
    SELECT student_id AS id, name AS student, grade AS score
    FROM students
    ORDER BY student_id
    LIMIT 5
""")

Naming the columns we want is clearer than `*` and slightly cheaper. `AS` renames a column in the result only — the stored data is untouched. Aliases such as `id`, `student`, and `score` make the output read better for whoever consumes it.

In [ ]:
# See which values a column really holds: DISTINCT removes duplicates.
show("SELECT DISTINCT course FROM students ORDER BY course")
show("SELECT DISTINCT status FROM students ORDER BY status")

`DISTINCT` collapses the 25 rows into the unique values, which is a quick way to learn what a column actually contains without scrolling the whole table. We see five courses and three statuses.

In [ ]:
# Step 5: filter rows with WHERE. Failing students (below 60), lowest first.
show("""
    SELECT student_id, name, course, grade
    FROM students
    WHERE grade < 60
    ORDER BY grade, student_id
""")

`WHERE` keeps only the rows for which a condition is true, so here we get the six students scoring below 60. Sorting by `grade` (ascending by default) and then by `student_id` puts the lowest score first and breaks any ties the same way every time.

In [ ]:
# The same filter on a text column, sorted best first with DESC.
show("""
    SELECT name, grade
    FROM students
    WHERE course = 'Computer Science'
    ORDER BY grade DESC, student_id
""")

Text values go in single quotes (`'Computer Science'`); double quotes in SQL mean an identifier, such as a column really named `"Order"`. `DESC` sorts from high to low, so the strongest Computer Science students come first.

In [ ]:
# Combine conditions with AND / OR. Parentheses matter: AND binds tighter than OR,
# so we group the two course names before applying the other filters.
show("""
    SELECT name, course, grade, status
    FROM students
    WHERE (course = 'Mathematics' OR course = 'Physics')
      AND status = 'active'
      AND grade >= 70
    ORDER BY course, grade DESC, student_id
""")

We combine conditions with `AND` and `OR`. The parentheses are not decoration: `AND` binds more tightly than `OR`, so without them the query would mean something different. Grouped this way, the condition reads "a Mathematics or Physics student who is also active and scoring at least 70".

In [ ]:
# A BOOLEAN column can be used directly as a condition: WHERE scholarship is
# true only for rows where scholarship = true.
show("SELECT name, grade FROM students WHERE scholarship ORDER BY grade DESC, student_id")

Because `scholarship` is a `BOOLEAN`, we can put it straight into `WHERE` — it is true only for the rows we want, and `NULL` would be treated as "not true". These six students are the scholarship holders.

In [ ]:
# Paginate with LIMIT and OFFSET: page 1 is the top five, page 2 skips five.
show("SELECT name, grade FROM students ORDER BY grade DESC, student_id LIMIT 5")
show("SELECT name, grade FROM students ORDER BY grade DESC, student_id LIMIT 5 OFFSET 5")

`LIMIT` caps the number of rows and `OFFSET` skips a number before it, together giving you pages of results. A large `OFFSET` gets slow because PostgreSQL still reads and discards the skipped rows — fine for a 25-row table, worth knowing as data grows.

In [ ]:
# The same filter, but driven by Python variables. The values travel as
# parameters, so even a hostile string is treated as plain text, not code.
course, min_grade = "Physics", 70
show("""
    SELECT name, grade
    FROM students
    WHERE course = %s AND grade >= %s
    ORDER BY grade DESC, student_id
""", (course, min_grade))

Here the values come from Python variables and are passed as parameters with `%s`, not pasted into the string. psycopg sends the SQL text and the values separately, so a value like `"x'; DROP TABLE students; --"` would simply be an unmatched text value. This is how we avoid SQL injection, the trick of hiding commands inside data.

In [ ]:
# Step 6: pattern matching. LIKE is case-sensitive; ILIKE ignores case.
# '%' matches any run of characters, '_' matches exactly one.
show("SELECT name FROM students WHERE name LIKE 'A%' ORDER BY name")
show("SELECT name FROM students WHERE name ILIKE '%AN%' ORDER BY name")

`LIKE` matches a text pattern, where `%` stands for any run of characters and `_` stands for exactly one. `ILIKE` is the case-insensitive version (a PostgreSQL extension). So `'A%'` finds names starting with a capital A, and `'%AN%'` finds any name containing "an" in any letter case.

In [ ]:
# IN is a tidy replacement for a chain of ORs.
show("""
    SELECT name, course, status
    FROM students
    WHERE course IN ('Biology', 'English')
      AND status IN ('inactive', 'graduated')
    ORDER BY name
""")

`IN (...)` checks whether a value is one of a short list and reads more cleanly than `course = 'Biology' OR course = 'English'`. Combined with the status filter, we get Biology or English students who are no longer active.

In [ ]:
# BETWEEN is inclusive on both ends, and works for numbers and dates alike.
show("SELECT name, grade FROM students WHERE grade BETWEEN 70 AND 79 ORDER BY grade, student_id")
show("""
    SELECT count(*) AS enrolled_sep_2_or_3
    FROM students
    WHERE enrollment_date BETWEEN '2024-09-02' AND '2024-09-03'
""")

`BETWEEN a AND b` includes both endpoints, so a grade of exactly 70 or exactly 79 is returned, and it works for dates too. One caution: for a column that stores a time of day, `BETWEEN day_start AND day_end` can drop the last day's later rows — prefer `>= start AND < next_day`.

In [ ]:
# Step 7: NULL handling. IS NULL finds missing values; "= NULL" finds nothing.
show("SELECT student_id, name FROM students WHERE email IS NULL ORDER BY student_id")
# The trap: "= NULL" is never true, so this returns no rows even though three do.
show("SELECT student_id, name FROM students WHERE email = NULL")

`NULL` means *no value / unknown* — it is not zero and not an empty string. SQL uses three-valued logic (true, false, unknown), and `WHERE` keeps only rows that are true. Because `email = NULL` evaluates to *unknown* rather than true, that query silently returns nothing; the right test is `IS NULL`. Three students (STU006, STU012, STU018) have no email on file.

In [ ]:
# count(*) counts rows; count(column) counts only non-NULL values, so the
# difference tells us how many emails are missing. COALESCE supplies a default.
show("SELECT count(*) AS total_rows, count(email) AS with_email FROM students")

show("""
    SELECT student_id, name, COALESCE(email, 'no email on file') AS contact
    FROM students
    WHERE student_id IN ('STU005', 'STU006', 'STU012')
    ORDER BY student_id
""")

`count(*)` counts rows, while `count(email)` counts only the rows where the email is not `NULL` — 25 versus 22, so three are missing. `COALESCE(email, 'no email on file')` returns the email when it exists and the fallback text when it does not, which is how we show a friendly default instead of a blank.

In [ ]:
# Step 8: label rows on the fly with CASE. Conditions are checked top to bottom,
# and the first match wins, so we list the strictest condition first.
show("""
    SELECT name, grade,
           CASE
               WHEN grade >= 90 THEN 'A'
               WHEN grade >= 80 THEN 'B'
               WHEN grade >= 70 THEN 'C'
               WHEN grade >= 60 THEN 'D'
               ELSE 'F'
           END AS letter
    FROM students
    ORDER BY grade DESC, student_id
    LIMIT 10
""")

- **Input:** every student's `grade` value.
- **What happens:** `CASE` checks the conditions from top to bottom and returns the value next to the first one that is true; `END` closes the expression, and `ELSE` covers everything left over.
- **Output:** a new `letter` column that appears only in the result — the stored grades are unchanged.
- **Why:** it lets us present data in a human-friendly form without adding or altering any columns.

Order matters: `>= 90` must be tested before `>= 80`, or a score of 95 would match the wrong branch.

```mermaid
graph TD
    G["grade"] --> C{"First true condition?"}
    C -->|">= 90"| A["'A'"]
    C -->|">= 80"| B["'B'"]
    C -->|">= 70"| Cn["'C'"]
    C -->|else| E["'D' or 'F'"]
    style G fill:#e1f5ff,stroke:#0288d1,color:#01579b
    style C fill:#fff9c4,stroke:#f9a825,color:#7f6000
    style A fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
    style B fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
    style Cn fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
    style E fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
```

In [ ]:
# The simple form "CASE column WHEN value" compares one column for equality,
# which reads nicely when the branches are exact matches.
show("""
    SELECT name, grade,
           CASE WHEN grade >= 60 THEN 'pass' ELSE 'fail' END AS result,
           CASE status WHEN 'active' THEN 'Enrolled'
                       WHEN 'inactive' THEN 'On leave'
                       ELSE 'Alumnus' END AS standing
    FROM students
    ORDER BY grade, student_id
    LIMIT 8
""")

There are two forms of `CASE`. The searched form (`CASE WHEN condition THEN ...`) tests full conditions, as above. The simple form (`CASE status WHEN 'active' THEN ...`) compares one column against exact values. If no branch matches and there is no `ELSE`, the result is `NULL`, so it is good practice to always add an `ELSE`.

In [ ]:
# Step 9: everyday built-in functions. Text first: upper, length, split_part,
# and '||' to glue strings together.
show("""
    SELECT upper(name)                AS name_upper,
           length(name)               AS chars,
           split_part(name, ' ', 1)    AS first_name,
           split_part(name, ' ', 2) || ', ' || split_part(name, ' ', 1) AS last_first
    FROM students
    ORDER BY student_id
    LIMIT 5
""")

PostgreSQL ships with hundreds of built-in functions. `upper` uppercases text, `length` counts characters, `split_part(name, ' ', 1)` returns the first word (the first name), and `||` joins strings together — here flipping "Alice Johnson" into "Johnson, Alice".

In [ ]:
# Math and casting. Beware integer division: 95 / 10 is 9, not 9.5.
show("""
    SELECT 95 / 10                      AS integer_division,
           95 / 10.0                    AS numeric_division,
           round(95 / 10.0, 1)          AS rounded,
           95::numeric / 10             AS cast_shortcut,
           CAST('42' AS integer) + 1    AS text_to_int
""")

When both sides of a division are integers, PostgreSQL does integer division and truncates the fraction — `95 / 10` is `9`. Making one side `numeric` (or writing `95::numeric`, the short cast) keeps the decimals, and `round(value, 1)` trims them to one place. The two spellings — `::numeric` and `CAST('42' AS integer)` — do the same job: convert a value from one type to another.

In [ ]:
# Dates: add days, subtract dates to get a number of days, format and extract
# parts. (We avoid current_date/now() so the output stays repeatable.)
show("""
    SELECT name,
           enrollment_date,
           enrollment_date + 30                         AS plus_30_days,
           to_char(enrollment_date, 'DD Mon YYYY')       AS pretty,
           extract(dow FROM enrollment_date)            AS weekday_no,
           graduation_date - enrollment_date            AS days_to_graduate
    FROM students
    WHERE status = 'graduated'
    ORDER BY student_id
""")

Adding a whole number to a `DATE` moves it forward that many days; subtracting two dates gives an integer number of days. `to_char` formats a date as text, and `extract` pulls out one part — here `dow` (day of week), which numbers Sunday as 0. We use a fixed `enrollment_date` instead of `current_date` so the output is the same every time you run it.

In [ ]:
# Step 10: put it all together. One query that uses column expressions,
# COALESCE, CASE, WHERE with AND and BETWEEN, ORDER BY and LIMIT.
show("""
    SELECT student_id,
           upper(split_part(name, ' ', 2)) || ', ' || split_part(name, ' ', 1) AS student,
           course,
           grade,
           CASE WHEN grade >= 90 THEN 'A' ELSE 'B' END AS letter,
           CASE WHEN scholarship THEN 'yes' ELSE 'no' END AS scholarship,
           COALESCE(email, 'no email on file') AS contact
    FROM students
    WHERE status = 'active' AND grade BETWEEN 80 AND 100
    ORDER BY grade DESC, student_id
    LIMIT 10
""")

This is the report the registrar wanted: active students scoring 80 or above, with names reformatted "Last, First", a letter grade, a readable scholarship flag, and a contact that never shows a blank. Every piece — `split_part`, `||`, `CASE`, `COALESCE`, `WHERE`, `BETWEEN`, `ORDER BY`, `LIMIT` — is something we met earlier in the lesson. `LIMIT 10` is only a safety cap here.

In [ ]:
# A one-row summary of the whole table using aggregate functions.
show("""
    SELECT count(*)                                         AS students,
           sum(CASE WHEN grade < 60 THEN 1 ELSE 0 END)      AS failing,
           sum(CASE WHEN scholarship THEN 1 ELSE 0 END)     AS scholarships,
           count(*) - count(email)                          AS missing_email,
           min(grade)                                       AS lowest,
           max(grade)                                       AS highest,
           round(avg(grade), 1)                             AS average
    FROM students
""")

`count`, `sum`, `min`, `max`, and `avg` summarize many rows into one. We use `sum(CASE WHEN ... THEN 1 ELSE 0 END)` as a counting trick: the `CASE` gives 1 for rows that match and 0 for the rest, and `sum` adds them up. `count(*) - count(email)` gives the number of missing emails.

In [ ]:
# Finally, close the connection. The table stays in the database for you to use.
conn.close()
print("connection closed")

Closing the connection frees the server resources we were holding. The `students` table itself stays in the `university` database, so you can reconnect and keep querying it whenever you like.